# Perceptron

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

In [ ]:
INPUT_SIZE: int = 2 # Constant for input size (n)
N_SAMPLES: int = 100

np.random.seed(42)

### Input

A perceptron takes in a series of $n$ numerical inputs $x_{1}, x_{2}, ..., x_{n}$

These might denote separate features eg $x_{1}$ = `stem_length`, $x_{2}$ = `sepal_width`, etc.

In [ ]:
X_train = np.random.uniform(-2, 2, size=(N_SAMPLES, INPUT_SIZE))

# Label 1 if x1 + x0 > 0 else 0
y_train = ((X_train[:, 0] + X_train[:, 1]) > 0).astype(int)

### Weights

A perceptron has a series of $n$ weights $w_{1}, w_{2}, ..., w_{n}$

Weights can be negative: a high positive weight means the feature strongly promotes activation, while a high negative weight means the feature works against it.

Weights are initialised as random values in $[-1, 1]$.

In [ ]:
class Perceptron:

    def __init__(self,
            input_size: int = INPUT_SIZE,
            learning_rate: float = 0.1
        ):
        self.input_size: int = input_size
        self.learning_rate: float = learning_rate
        self.initialize_weights()
        self.bias: float = 0

    def __str__(self):
        weights_str = ", ".join(f"{w:.3f}" for w in self.weights)
        lines = [
            '-' * 20,
            'Perceptron',
            '-' * 20,
            f"Weights: [{weights_str}]",
            f"Bias:    {self.bias:.3f}",
            '-' * 20,
        ]
        return "\n".join(lines)

    def initialize_weights(self):
        self.weights: np.array = np.random.uniform(-1, 1, size=self.input_size)

    def activation_function(self, z: float) -> int:
        return 1 if z >= 0 else 0

    def iterate(self, input: np.array) -> int:
        z: float = np.dot(input, self.weights) + self.bias
        return self.activation_function(z)

    def train(self, input: np.array, target: int):
        error = target - self.iterate(input)
        self.weights += self.learning_rate * error * input
        self.bias += self.learning_rate * error

In [ ]:
p: Perceptron = Perceptron(input_size=INPUT_SIZE)

weight_history = []
bias_history = []
accuracy_history = []

for epoch in range(20):
    for x, y in zip(X_train, y_train):
        p.train(x, y)
    weight_history.append(p.weights.copy())
    bias_history.append(p.bias)
    correct = sum(p.iterate(x) == y for x, y in zip(X_train, y_train))
    accuracy_history.append(correct / N_SAMPLES)

print(p)

In [ ]:
fig, (ax_main, ax_acc) = plt.subplots(1, 2, figsize=(12, 5))

# Scatter training data — fixed across all frames
colors = ['#e74c3c' if y == 0 else '#3498db' for y in y_train]
ax_main.scatter(X_train[:, 0], X_train[:, 1], c=colors, alpha=0.6, edgecolors='k', linewidths=0.3)
ax_main.set_xlim(-2.2, 2.2)
ax_main.set_ylim(-2.2, 2.2)
ax_main.set_xlabel("x₀")
ax_main.set_ylabel("x₁")
ax_main.axhline(0, color='grey', linewidth=0.5)
ax_main.axvline(0, color='grey', linewidth=0.5)

x_vals = np.linspace(-2.2, 2.2, 300)
boundary_line, = ax_main.plot([], [], 'k-', linewidth=2)
epoch_text = ax_main.set_title("Epoch 0")

# Accuracy subplot
ax_acc.set_xlim(0, len(weight_history))
ax_acc.set_ylim(0, 1.05)
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Accuracy")
ax_acc.axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
acc_line, = ax_acc.plot([], [], 'g-o', markersize=4)

def animate(i):
    w = weight_history[i]
    b = bias_history[i]

    # Decision boundary: w0*x0 + w1*x1 + b = 0  =>  x1 = -(w0*x0 + b) / w1
    if abs(w[1]) > 1e-6:
        y_vals = -(w[0] * x_vals + b) / w[1]
        boundary_line.set_data(x_vals, y_vals)
    else:
        boundary_line.set_data([], [])

    ax_main.set_title(f"Epoch {i + 1}  |  Accuracy: {accuracy_history[i]:.0%}")
    acc_line.set_data(range(1, i + 2), accuracy_history[:i + 1])
    return boundary_line, acc_line

anim = animation.FuncAnimation(
    fig, animate, frames=len(weight_history), interval=400, blit=True
)

plt.tight_layout()
HTML(anim.to_jshtml())